In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
select * from fq_dev_catalog.silver.branch_code_map

In [0]:
spark.sql("select * from fq_dev_catalog.gold.sales_orders_payload_1").groupBy(F.col("push_status")).count().display()

In [0]:
spark.sql("select * from fq_dev_catalog.gold.sales_orders_payload_1").filter(F.col("branch.name")=='ALB-133-Fujairah City Center').filter(F.col("id").like("%2026-10-29%")).display()

In [0]:
spark.sql("select * from fq_dev_catalog.gold.sales_orders_payload_1").filter(F.col("branch.name")=='ALB-140-Q1 Mall Al Warqaa').filter(F.col("id").like("%2026-10-28%")).display()

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_daily_performance where calendar_date = '2026-10-29' and deployment_name = 'Albaik - SQ DB35 - Fujairah city Center - 1005036'

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = "Incomplete"
where id like '%2026-10-28%' 

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_daily_performance where deployment_name like '' and calendar_date = '2026-10-26'

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_orders_payload_1 where branch.name = 'ALB-127-Dubai Silicon Oasis' and id like '%2026-10-25%' 

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1 
set branch.id = '33cabebc'
where branch.name = 'DEN-108-Business Bay Cloud Kitchen'

In [0]:
spark.sql("select * from fq_dev_catalog.gold.sales_orders_payload_1").groupBy(F.col("push_status")).count().display()

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Incomplete'
where branch.id = '33cabebc'

In [0]:
spark.sql('select * from fq_dev_catalog.gold.sales_orders_payload_1').filter(F.col('branch.name')=='ALB-127-Dubai Silicon Oasis').filter(F.col('id').like("%2026-10-25%")).display()

In [0]:
%sql
SELECT
  direct_qty,
  modifier_qty,
  direct_qty + modifier_qty AS total_qty
FROM (
  SELECT
    SUM(CASE WHEN i.id = '155309' THEN CAST(i.quantity AS DOUBLE) ELSE 0 END) AS direct_qty,
    SUM(aggregate(
          filter(i.modifiers, m -> m.id = '155309'),
          CAST(0 AS DOUBLE),
          (acc, m) -> acc + CAST(m.quantity AS DOUBLE))) AS modifier_qty
  FROM (
    SELECT explode(items) AS i
    FROM fq_dev_catalog.gold.sales_orders_payload_1
    WHERE salesTypeCode = 'DI'
      AND branch.name = 'ALB-127-Dubai Silicon Oasis'
      AND id LIKE '%2026-10-25%'
  ) t
) x;

In [0]:
%sql
Select * from fq_dev_catalog.gold.sales_orders_payload_1 where id like '%2026-10-23%' and branch.name = 'ALB-143-Deer Fields'

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Incomplete'
where push_status = 'Created'

In [0]:
spark.sql('select * from fq_dev_catalog.gold.sales_orders_payload_1').groupBy(F.col('push_status')).count().show()

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Incomplete'
where push_status = 'Created'

In [0]:

spark.sql('select * from fq_dev_catalog.gold.sales_orders_payload_1').filter(F.col('push_status').isNull()).display()

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Completed'
where push_status = 'Incomplete'

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Completed'
where push_status is null

In [0]:
%skip
spark.sql("""
    SELECT * 
    FROM fq_dev_catalog.gold.sales_orders_payload_1
""").select("push_status").distinct().show()

In [0]:
from pyspark.sql import functions as F

spark.sql("""
    SELECT * 
    FROM fq_dev_catalog.gold.sales_orders_payload_1
""").filter(F.col('push_status')=='Incomplete').display()

In [0]:
%sql
WITH items AS (
  SELECT
    right(id, 10) AS biz_date,
    item
  FROM fq_dev_catalog.gold.sales_orders_payload_1
  LATERAL VIEW explode(items) e AS item
  WHERE branch.name = 'ALB-100-Dubai Mall'
    AND to_date(right(id, 10)) = DATE'2026-10-08'
)
SELECT
  biz_date,
  -- 155205 sold directly as an item
  SUM(CASE WHEN item.id = '155205' THEN item.quantity ELSE 0 END) AS item_qty,
  -- 155205 included as a modifier inside combos
  SUM(aggregate(
        filter(item.modifiers, m -> m.id = '155205'),
        0D,
        (acc, m) -> acc + m.quantity
      )) AS modifier_qty,
  -- total
  SUM(CASE WHEN item.id = '155205' THEN item.quantity ELSE 0 END)
  + SUM(aggregate(
        filter(item.modifiers, m -> m.id = '155205'),
        0D,
        (acc, m) -> acc + m.quantity
      )) AS total_qty
FROM items
GROUP BY biz_date;

In [0]:
%sql
WITH items AS (
  SELECT
    right(id, 10) AS biz_date,
    salesTypeCode,
    item
  FROM fq_dev_catalog.gold.sales_orders_payload_1
  LATERAL VIEW explode(items) e AS item
  WHERE branch.name = 'ALB-127-Dubai Silicon Oasis'
    AND to_date(right(id, 10)) = DATE'2026-10-06'
)
SELECT
  biz_date,
  coalesce(salesTypeCode, 'TOTAL') AS sale_type,
  CASE salesTypeCode
    WHEN 'DLY' THEN 'Delivery'
    WHEN 'DI'  THEN 'Dine in'
    WHEN 'TA'  THEN 'Takeaway'
    ELSE 'All'
  END AS sale_type_name,
  -- 155205 sold directly as an item
  SUM(CASE WHEN item.id = '111133' THEN item.quantity ELSE 0 END) AS item_qty,
  -- 155205 included as a modifier inside combos
  SUM(aggregate(
        filter(item.modifiers, m -> m.id = '111133'),
        0D,
        (acc, m) -> acc + m.quantity
      )) AS modifier_qty,
  -- total
  SUM(CASE WHEN item.id = '111133' THEN item.quantity ELSE 0 END)
  + SUM(aggregate(
        filter(item.modifiers, m -> m.id = '111133'),
        0D,
        (acc, m) -> acc + m.quantity
      )) AS total_qty
FROM items
GROUP BY biz_date, ROLLUP(salesTypeCode)
ORDER BY biz_date, salesTypeCode IS NULL, salesTypeCode;

In [0]:
%sql
SELECT * FROM fq_dev_catalog.gold.sales_orders_payload_1 WHERE upper(branch.name) LIKE '%DUBAI MALL%'          -- or branch.name = '<exact branch name>'
    AND to_date(right(id, 10)) BETWEEN DATE'2026-10-08' AND DATE'2026-10-08'

In [0]:
%sql
WITH items AS (
  SELECT
    right(id, 10) AS biz_date,
    item
  FROM fq_dev_catalog.gold.sales_orders_payload_1
  LATERAL VIEW explode(items) e AS item
  WHERE upper(branch.name) LIKE '%DUBAI MALL%'          -- or branch.name = '<exact branch name>'
    AND to_date(right(id, 10)) BETWEEN DATE'2026-10-08' AND DATE'2026-10-08'
                      -- adjust field name
)
SELECT
  coalesce(biz_date, 'TOTAL') AS biz_date,
  SUM(item.quantity) AS item_qty
FROM items
GROUP BY ROLLUP(biz_date)
ORDER BY biz_date = 'TOTAL', biz_date;

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_daily_performance where calendar_date='2026-10-17'

In [0]:
df_store = spark.table("fq_dev_catalog.silver.dim_store").display()

In [0]:
from pyspark.sql import functions as F
spark.sql('select * from fq_dev_catalog.gold.sales_orders_payload_1').filter(F.col("placedAt").like("2026-10-11%")).filter(F.col('branch.name')== 'ALBAIK - SQ DB16 - DUBAI SILICON OASIS - 1005016').display()

In [0]:
from pyspark.sql import functions as F
spark.sql('select * from fq_dev_catalog.gold.sales_orders_payload_1').filter(F.col("placedAt").like("2026-10-08%")).filter(F.col('branch.name')== 'ALB-128-Karama').display()

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_orders_payload_1

In [0]:
from pyspark.sql import functions as F
spark.sql("select * from fq_dev_catalog.gold.sales_orders_payload_1").filter(F.col('id').like("%2026-10-07%")).filter(F.col("branch.name")=='ALB-113-Al Diyafah Satwa').display()

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Incomplete'
where branch.name = 'ALB-113-Al Diyafah Satwa' and id like '%2026-10-07%'

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Incomplete'
where branch.name = 'ALB-100-Dubai Mall'

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Completed'
where id like '%2026-10-19'

In [0]:
from pyspark.sql import functions as F

spark.sql("select * from fq_dev_catalog.gold.sales_daily_performance").filter(F.col("deployment_name")=="ALBAIK - SQ DB16 - DUBAI SILICON OASIS - 1005016").filter(F.col("calendar_date")=="2026-10-04").display()

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Incomplete'
where branch.name = 'ALB-128-Karama' and id like '%2026-10-08%'

In [0]:
from pyspark.sql import functions as F, Window


w = Window.partitionBy("ng_id").orderBy(F.col("load_datetime").desc())
TARGET_DATE = "2026-10-01"
TARGET_NG_ID = "0076f1c7-db01-b981-c51c-8ba9aa8a3fd0-a3fe27e3-8933-c2e52b6ddc9e-63716"

df_bronze = spark.table("fq_dev_catalog.bronze.sales_bills_2") \
    .filter(F.col("businessDate") == TARGET_DATE) \
    .filter(F.col("ng_id") == TARGET_NG_ID) \
    .filter(F.col("isVoid") == False) \
    .withColumn("rn", F.row_number().over(w)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .select("ng_id", "deployment_name", "businessDate", "_isDelivery", "_isTakeOut", "_tableId", "_kots")
df_bronze.display()

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_orders_payload_1 where id like '%2026-10-04' and branch.name = 'ALB-127-Dubai Silicon Oasis'

In [0]:
%sql
update fq_dev_catalog.gold.sales_orders_payload_1
set push_status = 'Incomplete'
where branch.name = 'ALB-127-Dubai Silicon Oasis' and id like '%2026-10-04%'
     

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_daily_performance where deployment_name = '' and calendar_date = ''

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_daily_performance where deployment_name like '%Dennys%' and calendar_date = '2026-10-26'

In [0]:
%sql
WITH p AS (
  SELECT salesTypeCode, items
  FROM fq_dev_catalog.gold.sales_orders_payload_1
  WHERE branch.name = 'ALB-128-Karama'
    AND right(id, 10) BETWEEN '2026-10-01' AND '2026-10-15'
),
it AS (
  SELECT salesTypeCode, item
  FROM p LATERAL VIEW explode(items) x AS item
)
SELECT
  SUM(item.quantity)                                              AS item_qty,
  SUM(aggregate(item.modifiers, 0D, (acc, m) -> acc + m.quantity)) AS modifier_qty,
  SUM(item.quantity)
    + SUM(aggregate(item.modifiers, 0D, (acc, m) -> acc + m.quantity)) AS total_qty
FROM it;

In [0]:
%sql
WITH p AS (
  SELECT salesTypeCode, items
  FROM fq_dev_catalog.gold.sales_orders_payload_1
  WHERE branch.name = 'ALB-130-Al Shamkha ADNOC'
    AND right(id, 10) = '2026-10-01'
),
it AS (
  SELECT salesTypeCode, item
  FROM p LATERAL VIEW explode(items) x AS item
)
SELECT
  SUM(item.quantity) AS item_qty,
  SUM(aggregate(item.modifiers, 0D, (acc, m) -> acc + m.quantity)) AS modifier_qty,
  SUM(item.quantity)
    + SUM(aggregate(item.modifiers, 0D, (acc, m) -> acc + m.quantity)) AS total_qty
FROM it;



In [0]:
%sql
SELECT * FROM fq_dev_pnl_catalog.silver.pnl_budget_flat_data WHERE  brand_id = 'DENNYS' and year = 2026 and month = 'AUG' 

In [0]:
%sql
SELECT * FROM fq_dev_pnl_catalog.silver.management_pnl WHERE  brand_id = 'DENNYS' and year = 2026 and month >= 'JAN'

In [0]:
%sql
SELECT * FROM fq_dev_catalog.gold.management_pnl WHERE  brand_id in ('DENNYS') and year = 2026 and month in('AUG')

In [0]:
%sql
SELECT * FROM fq_dev_catalog.gold.management_pnl WHERE  brand_id in ('DENNYS', 'AI') and year = 2026 and month in('JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG')

In [0]:
from pyspark.sql import functions as F

START, END = "2026-01-01", "2026-10-01"  # end exclusive

item = (
    spark.table("fq_dev_catalog.gold.sales_item_wise")
    .filter((F.col("business_date") >= START) & (F.col("business_date") < END))
    .groupBy("deployment_name", F.date_format("business_date", "yyyy-MM").alias("month"))
    .agg(F.sum("net_item_amount_after_discount").alias("item_total"))
)

daily = (
    spark.table("fq_dev_catalog.gold.sales_daily_performance")
    .filter((F.col("calendar_date") >= START) & (F.col("calendar_date") < END))
    .groupBy("deployment_name", F.date_format("calendar_date", "yyyy-MM").alias("month"))
    .agg(F.sum("net_sales_after_discount").alias("daily_total"))
)

result = (
    item.join(daily, ["deployment_name", "month"], "full_outer")
    .withColumn("diff", F.round(F.coalesce("item_total", F.lit(0)) - F.coalesce("daily_total", F.lit(0)), 3))
    .withColumn("status", F.when(F.abs("diff") < 1, "MATCH").otherwise("MISMATCH"))
    .orderBy("deployment_name", "month")
)

display(result)
# only problem rows:
# display(result.filter("status = 'MISMATCH'"))

In [0]:
from pyspark.sql import functions as F

START, END = "2026-03-15", "2026-03-31"  # end exclusive

item = (
    spark.table("fq_dev_catalog.gold.sales_item_wise")
    .filter((F.col("deployment_name") == "Dennys ADM") &
            (F.col("business_date") == START))
    .groupBy(F.date_format("business_date", "yyyy-MM").alias("month"))
    .agg(F.sum("net_item_amount_after_discount").alias("item_total"))
)

daily = (
    spark.table("fq_dev_catalog.gold.sales_daily_performance")
    .filter((F.col("deployment_name") == "Dennys ADM") &
            (F.col("calendar_date") == START))
    .groupBy(F.date_format("calendar_date", "yyyy-MM").alias("month"))
    .agg(F.sum("net_sales_after_discount").alias("daily_total"))
)

result = (
    item.join(daily, "month", "full_outer")
    .withColumn("diff", F.round(F.coalesce("item_total", F.lit(0)) - F.coalesce("daily_total", F.lit(0)), 3))
    .withColumn("status", F.when(F.abs("diff") < 1, "MATCH").otherwise("MISMATCH"))
    .orderBy("month")
)

display(result)

In [0]:
from pyspark.sql import functions as F

START, END = "2026-03-01", "2026-03-31"  # end exclusive
df = spark.sql("select distinct deployment_name from fq_dev_catalog.gold.sales_item_wise")


item = (
    spark.table("fq_dev_catalog.gold.sales_item_wise")
    .filter((F.col("deployment_name") == "Dennys ADM") &
            (F.col("business_date") >= START) & (F.col("business_date") < END))
    .groupBy(F.date_format("business_date", "yyyy-MM").alias("month"))
    .agg(F.sum("net_item_amount_after_discount").alias("item_total"))
)

daily = (
    spark.table("fq_dev_catalog.gold.sales_daily_performance")
    .filter((F.col("deployment_name") == "Dennys ADM") &
            (F.col("calendar_date") >= START) & (F.col("calendar_date") < END))
    .groupBy(F.date_format("calendar_date", "yyyy-MM").alias("month"))
    .agg(F.sum("net_sales_after_discount").alias("daily_total"))
)

result = (
    item.join(daily, "month", "full_outer")
    .withColumn("diff", F.round(F.coalesce("item_total", F.lit(0)) - F.coalesce("daily_total", F.lit(0)), 3))
    .withColumn("status", F.when(F.abs("diff") < 1, "MATCH").otherwise("MISMATCH"))
    .orderBy("month")
)

display(result)

In [0]:
%sql
select sum(net_item_amount_after_discount) from fq_dev_catalog.gold.sales_item_wise where deployment_name = 'Dennys ADM' and business_date >= '2026-01-01' and business_date <= '2026-01-31'

In [0]:
%sql
SELECT * FROM fq_dev_catalog.gold.sales_daily_performance  WHERE calendar_date = '2026-10-14'

In [0]:

%sql
SELECT
    deployment_name,
    SUM(CASE WHEN _isDelivery = TRUE THEN 1 ELSE 0 END) AS delivery_bills,
    SUM(
        CASE
            WHEN (_isDelivery IS NULL OR _isDelivery = FALSE)
                 AND _isTakeOut = TRUE
            THEN 1
            ELSE 0
        END
    ) AS takeaway_bills,
    SUM(
        CASE
            WHEN (_isDelivery IS NULL OR _isDelivery = FALSE)
                 AND (_isTakeOut IS NULL OR _isTakeOut = FALSE)
            THEN 1
            ELSE 0
        END
    ) AS dine_in_bills
FROM (
    SELECT DISTINCT
        ng_id,
        deployment_name,
        _isDelivery,
        _isTakeOut
    FROM fq_dev_catalog.bronze.sales_bills_2
    WHERE businessDate = '2026-10-01'
      AND isVoid = FALSE
) AS bills
GROUP BY deployment_name
ORDER BY dine_in_bills DESC, deployment_name;

In [0]:
%sql
WITH p AS (
    SELECT
        salesTypeCode,
        items
    FROM fq_dev_catalog.gold.sales_orders_payload_1
    WHERE branch.name = 'ALB-116-Al Nahda'
),

it AS (
    SELECT
        salesTypeCode,
        item
    FROM p
    LATERAL VIEW EXPLODE(items) x AS item
),

a AS (
    -- Item-level calculations
    SELECT
        salesTypeCode,
        CAST(item.id AS STRING) AS code,
        'item' AS kind,
        item.quantity AS qty,
        item.quantity * item.unitPrice.excludingTax AS subtotal,
        item.quantity * item.unitPrice.discountAmount AS discount,
        item.quantity * item.unitPrice.includingTax AS gross
    FROM it

    UNION ALL

    -- Modifier-level calculations
    SELECT
        salesTypeCode,
        CAST(m.id AS STRING) AS code,
        'mod' AS kind,
        m.quantity AS qty,
        m.quantity * m.unitPrice.excludingTax AS subtotal,
        m.quantity * m.unitPrice.discountAmount AS discount,
        m.quantity * m.unitPrice.includingTax AS gross
    FROM it
    LATERAL VIEW EXPLODE(item.modifiers) y AS m
)

SELECT
    salesTypeCode,
    code,
    ROUND(SUM(qty), 3) AS qty,
    ROUND(SUM(subtotal), 3) AS subtotal,
    ROUND(SUM(discount), 3) AS discount,
    ROUND(SUM(gross), 3) AS gross,
    ROUND(
        SUM(CASE WHEN kind = 'item' THEN qty ELSE 0 END),
        3
    ) AS item_qty
FROM a
GROUP BY
    salesTypeCode,
    code
ORDER BY
    salesTypeCode,
    code;

In [0]:
%sql
SELECT * FROM fq_dev_catalog.silver.sales_items_silver_2
WHERE business_date = '2026-01-01' limit 20

In [0]:
%sql
DESCRIBE DETAIL fq_dev_catalog.silver.sales_items_silver_2

In [0]:
%sql
select * from fq_dev_pnl_catalog.bronze.pnl_actual_flat_data where Year = 2025 and `Store Name` = 'DEN FPZ'


In [0]:
%sql
-- Step 1: Discover all distinct brand_id values in the table
SELECT DISTINCT brand_id FROM fq_dev_catalog.gold.management_pnl ORDER BY brand_id

In [0]:
%sql
SELECT * FROM fq_dev_catalog.gold.management_pnl WHERE  brand_id = 'DENNYS' and year = 2026 and month = 'SEP'

In [0]:
%sql
SELECT * FROM fq_dev_catalog.gold.management_pnl WHERE brand_id IS NOT NULL

In [0]:
%sql
 select * from fq_dev_pnl_catalog.silver.management_pnl where where brand_id = 'AI' and year = '2026'and budget_amount >0  --and 'netsuite_location_name' = "DEN-105-Deira City Center"

In [0]:
%sql
select * from fq_dev_catalog.gold.management_pnl where brand_id = 'AI' and year = '2026' and budget_amount >0 --and month = 'APR'

In [0]:
# Databricks notebook source
# MAGIC %sql
# MAGIC select * from fq_dev_catalog.bronze.version_control

# COMMAND ----------

# SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
# VERSION_TABLE = "fq_dev_catalog.bronze.version_control"
# CONSUMER_NAME = "new_billwise_itemwise_fact"

# COMMAND ----------

# spark.sql(f"""
# INSERT INTO {VERSION_TABLE}
# Values
# ('{SOURCE_TABLE}', '{CONSUMER_NAME}', '{2216}', current_timestamp())
# """)
 

# COMMAND ----------

# from delta.tables import DeltaTable
# from pyspark.sql.functions import col, max as _max

# # ------------------------------
# # 1 Configuration
# # ------------------------------
# SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
# VERSION_TABLE = "fq_dev_catalog.bronze.version_control"
# CONSUMER_NAME = "new_billwise_itemwise_fact"   # unique per notebook

# # ------------------------------
# # 2 Get latest version from source table
# # ------------------------------
# delta_table = DeltaTable.forName(spark, SOURCE_TABLE)

# latest_version = (
#     delta_table.history()
#     .select(_max("version").alias("latest_version"))
#     .collect()[0]["latest_version"]
# )

# # ------------------------------
# # 3 Get last processed version from VERSION_TABLE
# # ------------------------------
# if spark.catalog.tableExists(VERSION_TABLE):
#     last_processed_version = (
#         spark.table(VERSION_TABLE)
#         .filter(col("source_table") == SOURCE_TABLE)
#         .filter(col("consumer_name") == CONSUMER_NAME)
#         .agg(_max("last_processed_version").alias("last_version"))
#         .collect()[0]["last_version"]
#     )
# else:
#     last_processed_version = None

# last_processed_version = (
#     last_processed_version if last_processed_version is not None else -1
# )

# # ------------------------------
# # 4 Decide start_version
# # ------------------------------
# if last_processed_version >= latest_version:
#     print(f"No new versions to process. Latest={latest_version}, Last={last_processed_version}")
#     start_version = None
# else:
#     start_version = last_processed_version + 1
#     print(f"Processing versions {start_version} → {latest_version}")

# ------------------------------
# 5️ Always create incremental_df (NEVER None)
# ------------------------------
# if start_version is None:
#     incremental_df = (
#         spark.read.table(SOURCE_TABLE)
#         .limit(0)   # ✅ empty DF with same schema
#     )
# else:
#     incremental_df = (
#         spark.read.format("delta")
#         .option("readChangeFeed", "true")
#         .option("startingVersion", start_version)
#         .option("endingVersion", latest_version)
#         .table(SOURCE_TABLE)
#         .filter(col("_change_type").isin("insert"))
#     )

# ------------------------------
# 6️ Safe to use everywhere
# ------------------------------
# incremental_df = incremental_df.filter(col("isVoid") == False)

# display(incremental_df)

# COMMAND ----------

# incremental_df = spark.read.table("fq_dev_catalog.bronze.sales_bills_2")


# COMMAND ----------

# DBTITLE 1,Incremental Version Tracking and Sales Bills Data Prepa ...
# Databricks notebook source
# MAGIC %sql
# MAGIC select * from fq_dev_catalog.bronze.version_control

# COMMAND ----------

# SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
# VERSION_TABLE = "fq_dev_catalog.bronze.version_control"
# CONSUMER_NAME = "new_billwise_itemwise_fact"

# COMMAND ----------

# spark.sql(f"""
# INSERT INTO {VERSION_TABLE}
# Values
# ('{SOURCE_TABLE}', '{CONSUMER_NAME}', '{2216}', current_timestamp())
# """)
 

# COMMAND ----------

# from delta.tables import DeltaTable
# from pyspark.sql.functions import col, max as _max

# # ------------------------------
# # 1️⃣ Configuration
# # ------------------------------
# SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
# VERSION_TABLE = "fq_dev_catalog.bronze.version_control"
# CONSUMER_NAME = "new_billwise_itemwise_fact"   # unique per notebook

# # ------------------------------
# # 2️⃣ Get latest version from source table
# # ------------------------------
# delta_table = DeltaTable.forName(spark, SOURCE_TABLE)

# latest_version = (
#     delta_table.history()
#     .select(_max("version").alias("latest_version"))
#     .collect()[0]["latest_version"]
# )

# # ------------------------------
# # 3️⃣ Get last processed version from VERSION_TABLE
# # ------------------------------
# if spark.catalog.tableExists(VERSION_TABLE):
#     last_processed_version = (
#         spark.table(VERSION_TABLE)
#         .filter(col("source_table") == SOURCE_TABLE)
#         .filter(col("consumer_name") == CONSUMER_NAME)
#         .agg(_max("last_processed_version").alias("last_version"))
#         .collect()[0]["last_version"]
#     )
# else:
#     last_processed_version = None

# last_processed_version = (
#     last_processed_version if last_processed_version is not None else -1
# )

# # ------------------------------
# # 4️⃣ Decide start_version
# # ------------------------------
# if last_processed_version >= latest_version:
#     print(f"No new versions to process. Latest={latest_version}, Last={last_processed_version}")
#     start_version = None
# else:
#     start_version = last_processed_version + 1
#     print(f"Processing versions {start_version} → {latest_version}")

# # ------------------------------
# # 5️⃣ Always create incremental_df (NEVER None)
# # ------------------------------
# if start_version is None:
#     incremental_df = (
#         spark.read.table(SOURCE_TABLE)
#         .limit(0)   # ✅ empty DF with same schema
#     )
# else:
#     incremental_df = (
#         spark.read.format("delta")
#         .option("readChangeFeed", "true")
#         .option("startingVersion", start_version)
#         .option("endingVersion", latest_version)
#         .table(SOURCE_TABLE)
#         .filter(col("_change_type").isin("insert"))
#     )

# # ------------------------------
# # 6️⃣ Safe to use everywhere
# # ------------------------------
# incremental_df = incremental_df.filter(col("isVoid") == False)

# display(incremental_df)

# COMMAND ----------

# from pyspark.sql.functions import col
incremental_df = spark.read.table("fq_dev_catalog.bronze.sales_bills_2")

# COMMAND ----------

# Databricks notebook source
from pyspark.sql.functions import col, lit, expr, when, explode, size, to_date, to_timestamp, coalesce

# Read source bronze table, dedupe by ng_id
df = (
    incremental_df
    .filter(col("isVoid") == False)
    .dropDuplicates(subset=["ng_id"])
)

# Explode KOTs -> items
items_df = (
    df.select(
        col("ng_id").alias("sales_bill_id"), col("deployment_name"),
        to_date(col("businessDate")).alias("business_date"),
        col("tabType").alias("order_type"), col("tab").alias("tab_name"),
        to_timestamp(col("_closeTime")).alias("close_time"),
        to_timestamp(col("_created")).alias("created_time"),
        explode(col("_kots")).alias("kot")
    )
    .select(
        "sales_bill_id", col("kot.kotNumber").alias("kot_number"),
        "deployment_name", "business_date", "order_type", "tab_name",
        "close_time", "created_time",
        explode(col("kot.items")).alias("item")
    )
)


base_items_df = (
    items_df.filter(col("kot.isVoid") == False)
    .select(
        "sales_bill_id", "kot_number",
        col("item.itemIndex").alias("item_index"),
        col("item.displayName").alias("item_name"),
        col("item.category.categoryName").alias("category_name"),
        col("item.category.supercategory.supercategoryName").alias("supercategory_name"),
        col("item.number").alias("item_upccode"),
        col("item.quantity").cast("int").alias("quantity"),
        col("item.rate").cast("decimal(18,3)").alias("unit_price"),
        (
            coalesce(expr("aggregate(item.discounts.discountAmount, 0D, (acc, x) -> acc + x)"), lit(0.0))
            + coalesce(col("item.billDiscountAmount"), lit(0.0))
        ).cast("decimal(18,3)").alias("item_discount_amount"),
        when(col("item.discounts").isNotNull() & (expr("size(item.discounts)") > 0),
             expr("array_join(item.discounts.offerName, ', ')"))   # all offer names, comma-separated
        .otherwise(lit(None).cast("string"))
        .alias("item_discount_name"),
        expr("""
            concat(
                coalesce(
                    transform(item.discounts, x -> named_struct(
                        'offer_name', x.offerName,
                        'discount_amount', CAST(x.discountAmount AS DECIMAL(18,3))
                    )),
                    array()
                ),
                CASE WHEN item.billDiscountAmount IS NOT NULL AND item.billDiscountAmount != 0
                     THEN array(named_struct(
                            'offer_name', CAST(NULL AS STRING),
                            'discount_amount', CAST(item.billDiscountAmount AS DECIMAL(18,3))
                          ))
                     ELSE CAST(array() AS ARRAY<STRUCT<offer_name:STRING, discount_amount:DECIMAL(18,3)>>)
                END
            )
        """).alias("item_discount_bifurcate"),   # full breakdown: every discount entry + bill discount, each kept separate
        col("item.subtotal").cast("decimal(18,3)").alias("net_item_amount"),
        expr("aggregate(item.taxes.tax_amount, 0D, (acc, x) -> acc + x)")
            .cast("decimal(18,3)").alias("tax_amount"),
        col("item.subTotalWithTax").cast("decimal(18,3)").alias("gross_item_amount"),
        "deployment_name", "business_date", "close_time", "created_time",
        lit(None).cast("string").alias("parent_upcode"), "order_type", "tab_name",
        lit(True).alias("is_base_item"),
        col("item.category.isCombos").alias("is_combo_item"),
        col("item.category.isAddons").alias("is_addon_item"),
        lit(False).alias("is_combo_child_item"), lit(False).alias("is_addon_child_item"),
        lit("POSIST").alias("source_system")
    )
)

addon_items_df = (
    items_df.filter((col("kot.isVoid") == False) & col("item.addOns").isNotNull()
                     & (size(col("item.addOns")) > 0))
    .select("sales_bill_id", "kot_number", "deployment_name", "business_date",
            "close_time", "created_time", "order_type", "tab_name",
            col("item.number").alias("parent_upcode"),
            col("item.itemIndex").alias("item_index"),
            explode(col("item.addOns")).alias("addon"))
    .select(
        "sales_bill_id", "kot_number", "item_index",
        col("addon.displayName").alias("item_name"),
        col("addon.category.categoryName").alias("category_name"),
        col("addon.category.supercategory.supercategoryName").alias("supercategory_name"),
        col("addon.number").alias("item_upccode"),
        col("addon.quantity").cast("int").alias("quantity"),
        col("addon.rate").cast("decimal(18,3)").alias("unit_price"),
        col("addon.subtotal").cast("decimal(18,3)").alias("net_item_amount"),
        expr("aggregate(addon.taxes.tax_amount, 0D, (acc, x) -> acc + x)")
            .cast("decimal(18,3)").alias("tax_amount"),
        col("addon.subTotalWithTax").cast("decimal(18,3)").alias("gross_item_amount"),
        col("addon.billDiscountAmount")
            .cast("decimal(18,3)").alias("item_discount_amount"),
        lit(None).cast("string").alias("item_discount_name"),
        when(
            col("addon.billDiscountAmount").isNotNull() & (col("addon.billDiscountAmount") != 0),
            expr("array(named_struct('offer_name', CAST(NULL AS STRING), 'discount_amount', CAST(addon.billDiscountAmount AS DECIMAL(18,3))))")
        ).otherwise(
            expr("CAST(array() AS ARRAY<STRUCT<offer_name:STRING, discount_amount:DECIMAL(18,3)>>)")
        ).alias("item_discount_bifurcate"),   # addon.discounts is always null/empty in practice, so this mirrors billDiscountAmount only
        "deployment_name", "business_date", "close_time", "created_time", "parent_upcode",
        "order_type", "tab_name",
        lit(False).alias("is_base_item"), lit(False).alias("is_combo_item"),
        lit(False).alias("is_addon_item"), lit(False).alias("is_combo_child_item"),
        lit(True).alias("is_addon_child_item"), lit("POSIST").alias("source_system")
    )
)

combo_child_df = (
    items_df
    .filter(
        (col("kot.isVoid") == False) &
        col("item.mapComboItems").isNotNull() &
        (size(col("item.mapComboItems")) > 0)
    )
    .select(
        col("sales_bill_id"),
        col("kot_number"),
        col("deployment_name"),
        col("business_date"),
        col("close_time"),
        col("created_time"),
        col("order_type"),
        col("tab_name"),
        col("item.itemIndex").alias("item_index"),
        col("item.number").alias("parent_upcode"),
        col("item.quantity").cast("int").alias("parent_quantity"),
        explode(col("item.mapComboItems")).alias("combo")
    )
    .select(
        col("sales_bill_id"),
        col("kot_number"),
        col("item_index"),
        col("combo.displayName").alias("item_name"),
        col("combo.category.categoryName").alias("category_name"),
        col("combo.category.supercategory.supercategoryName").alias("supercategory_name"),
        col("combo.number").alias("item_upccode"),

        # ✅ quantity = combo × parent
        (
            col("combo.quantity").cast("int") *
            col("parent_quantity")
        ).alias("quantity"),

        col("combo.rate").cast("decimal(18,3)").alias("unit_price"),

        # 🔥 discount always 0 for combo child rows (discount lives on the parent combo item)
        lit(0).cast("decimal(18,3)").alias("item_discount_amount"),
        lit(None).cast("string").alias("item_discount_name"),
        expr("CAST(array() AS ARRAY<STRUCT<offer_name:STRING, discount_amount:DECIMAL(18,3)>>)")
            .alias("item_discount_bifurcate"),

        # 🔥 IMPORTANT: zero amounts to avoid double counting
        lit(0).cast("decimal(18,3)").alias("net_item_amount"),
        lit(0).cast("decimal(18,3)").alias("tax_amount"),
        lit(0).cast("decimal(18,3)").alias("gross_item_amount"),

        col("deployment_name"),
        col("business_date"),
        col("close_time"),
        col("created_time"),
        col("parent_upcode"),
        col("order_type"),
        col("tab_name"),
        lit(False).alias("is_base_item"),
        col("combo.category.isCombos").alias("is_combo_item"),
        col("combo.category.isAddons").alias("is_addon_item"),
        lit(True).alias("is_combo_child_item"),
        lit(False).alias("is_addon_child_item"),
        lit("POSIST").alias("source_system")
    )
)

all_items_df = (
    base_items_df
    .unionByName(addon_items_df.select(*base_items_df.columns))      # column order/name aligned union
    .unionByName(combo_child_df.select(*base_items_df.columns))      # add combo child rows
)

from pyspark.sql import functions as F

# Recompute is_base_item now that combo-child rows are present:
# a row is a true base item only if it's not a combo, not an addon,
# and not a child of either a combo or an addon.
all_items_df = all_items_df.withColumn(
    "is_base_item",
    F.when(
        (F.col("is_combo_item") == False) &
        (F.col("is_addon_item") == False) &
        (F.col("is_addon_child_item") == False) &
        (F.col("is_combo_child_item") == False),
        True
    ).otherwise(False)
)

# COMMAND ----------

# Item-
df_items = all_items_df.select(
    col("sales_bill_id").alias("ng_id"),
    "kot_number", "item_index", "deployment_name", "business_date",
    "order_type", "tab_name", "close_time", "created_time",
    "item_upccode", "quantity", "unit_price", "net_item_amount",
    col("tax_amount").alias("item_tax_amount"),
    "gross_item_amount", "item_discount_amount", "item_discount_name",
    "item_discount_bifurcate",   # NEW
    "parent_upcode",
    "is_base_item", "is_combo_item", "is_addon_item",
    "is_combo_child_item", "is_addon_child_item",
    "category_name", "item_name", "supercategory_name", "source_system"
)

# Fill missing discounts with 0 so downstream arithmetic doesn't null out
df_items = df_items.fillna(value=0.000, subset=["item_discount_amount"])

# Recompute net/gross amounts around the (now reliable) item_discount_amount
df_items = (
    df_items
    .withColumn("net_item_amount_after_discount", col("net_item_amount") - col("item_discount_amount"))
    .withColumn("gross_item_amount", col("net_item_amount_after_discount") + col("item_tax_amount"))
    .withColumnRenamed("net_item_amount", "net_item_amount_before_discount")
)

# COMMAND ----------

from pyspark.sql.functions import from_utc_timestamp, to_timestamp, when

# Extract online order source fields from bronze
df_online_bills = (
    df.filter(col("onlineBill").isNotNull())
    .select("ng_id", "onlineBill")
    .withColumn("source_online_name", col("onlineBill.source.name"))
    .withColumn("source_online_id", col("onlineBill.source.id"))
    .withColumn("source_online_order_id", col("onlineBill.source.order_id"))
)

# ------------------------------
# Bill-level payments (from bronze `payments`)
#   payments.cards[] -> cardType, totalAmount, detail[] (onlineType, onlineName, amount)
#   payments.cash[]  -> cash entries
# One bill can have several payments, so they are kept as aligned ARRAYS
# (same position in type / name / amount = same payment). This avoids
# duplicating item rows, which would break the merge key and double count sums.
# ------------------------------
from pyspark.sql.functions import explode_outer, collect_list, struct, lit

# Card / online payments: one row per detail entry; if a card type has no detail
# but totalAmount > 0, fall back to the card-level amount.
card_payments = (
    df.select("ng_id", explode(col("payments.cards")).alias("card"))
    .select(
        "ng_id",
        col("card.cardType").alias("pay_type"),
        col("card.totalAmount").alias("card_total"),
        explode_outer(col("card.detail")).alias("d")
    )
    .filter(col("d").isNotNull() | (col("card_total") > 0))
    .select(
        "ng_id",
        col("pay_type"),
        coalesce(col("d.onlineName"), col("pay_type")).alias("pay_name"),
        coalesce(col("d.amount"), col("card_total")).cast("decimal(18,3)").alias("pay_amount")
    )
)

# Cash payments (payments.cash is [] in the sample, so the element field name
# `amount` is an assumption - confirm with df.select("payments.cash").printSchema())
# payments.cash is ARRAY<DOUBLE> - each element is the cash amount itself
cash_payments = (
    df.select("ng_id", explode(col("payments.cash")).alias("c"))
    .filter(col("c").isNotNull() & (col("c") > 0))   # skip empty/zero cash entries
    .select(
        "ng_id",
        lit("Cash").alias("pay_type"),
        lit("Cash").alias("pay_name"),
        col("c").cast("decimal(18,3)").alias("pay_amount")
    )
)

df_payments = (
    card_payments.unionByName(cash_payments)
    .groupBy("ng_id")
    .agg(collect_list(struct("pay_type", "pay_name", "pay_amount")).alias("p"))
    .select(
        "ng_id",
        col("p.pay_type").alias("bill_payments_type"),
        col("p.pay_name").alias("bill_payments_name"),
        col("p.pay_amount").alias("bill_payments_amount")
    )
)

df_bills = (
    spark.read.table("fq_dev_catalog.silver.sales_bills")
    .join(df_online_bills.select("ng_id", "source_online_name", "source_online_order_id"),
          on="ng_id", how="left")
    .join(df_payments, on="ng_id", how="left")
    .drop("sys_id", "business_date", "deployment_name", "order_type",
          "tab_name", "source_system", "close_time")
)

df_items_bills = df_items.join(df_bills, on="ng_id", how="left")

df_items_bills_final = (
    df_items_bills
    .withColumn("created_time", from_utc_timestamp(to_timestamp(col("created_time")), "Asia/Dubai"))
    .withColumn("close_time", from_utc_timestamp(to_timestamp(col("close_time")), "Asia/Dubai"))
    .withColumn("source_system", when(col("source_online_name").isNotNull(), "ONLINE").otherwise(col("source_system")))
)

# COMMAND ----------

df_final = df_items_bills_final.select(
    "deployment_name", "business_date", "company_id", "country_code", "ng_id", "bill_id",
    "kot_number",  "source_system", "source_online_name", "source_online_order_id",
    "item_upccode", "quantity", "unit_price",
    "net_item_amount_before_discount", "item_discount_amount", "item_discount_name",
    "item_discount_bifurcate",   # NEW
    "net_item_amount_after_discount", "item_tax_amount", "gross_item_amount",
    "parent_upcode", "order_type", "tab_name",
    "is_base_item", "is_combo_item", "is_addon_item", "is_combo_child_item", "is_addon_child_item",
    "category_name", "item_name", "supercategory_name", "created_time", "close_time",
    "kots_ids",
    col("net_amount_before_discount").alias("bill_net_amount_before_discount"),
    "bill_discount", "total_items_discount",
    col("net_amount_after_discount").alias("bill_net_amount_after_discount"),
    col("tax_amount").alias("bill_tax_amount"),
    col("gross_amount").alias("bill_gross_amount"),
    "total_items", "is_void", "void_reason", "covers",
    "bill_payments_type", "bill_payments_name", "bill_payments_amount"   # NEW
)

df_final.display()





# COMMAND ----------

df_final.printSchema()

# COMMAND ----------

# df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fq_dev_catalog.gold.itemwise_billwise_fact")

# COMMAND ----------

# # Databricks notebook source
# from pyspark.sql.functions import col, lit, expr, when, explode, size, to_date, to_timestamp, coalesce

# # Read source bronze table, dedupe by ng_id
# df = (
#     incremental_df
#     .filter(col("isVoid") == False)
#     .dropDuplicates(subset=["ng_id"])
# )

# # Explode KOTs -> items
# items_df = (
#     df.select(
#         col("ng_id").alias("sales_bill_id"), col("deployment_name"),
#         to_date(col("businessDate")).alias("business_date"),
#         col("tabType").alias("order_type"), col("tab").alias("tab_name"),
#         to_timestamp(col("_closeTime")).alias("close_time"),
#         to_timestamp(col("_created")).alias("created_time"),
#         explode(col("_kots")).alias("kot")
#     )
#     .select(
#         "sales_bill_id", col("kot.kotNumber").alias("kot_number"),
#         "deployment_name", "business_date", "order_type", "tab_name",
#         "close_time", "created_time",
#         explode(col("kot.items")).alias("item")
#     )
# )


# base_items_df = (
#     items_df.filter(col("kot.isVoid") == False)
#     .select(
#         "sales_bill_id", "kot_number",
#         col("item.itemIndex").alias("item_index"),
#         col("item.displayName").alias("item_name"),
#         col("item.category.categoryName").alias("category_name"),
#         col("item.category.supercategory.supercategoryName").alias("supercategory_name"),
#         col("item.number").alias("item_upccode"),
#         col("item.quantity").cast("int").alias("quantity"),
#         col("item.rate").cast("decimal(18,3)").alias("unit_price"),
#         (
#             coalesce(expr("aggregate(item.discounts.discountAmount, 0D, (acc, x) -> acc + x)"), lit(0.0))
#             + coalesce(col("item.billDiscountAmount"), lit(0.0))
#         ).cast("decimal(18,3)").alias("item_discount_amount"),
#         when(col("item.discounts").isNotNull() & (expr("size(item.discounts)") > 0),
#              expr("array_join(item.discounts.offerName, ', ')"))   # all offer names, comma-separated
#         .otherwise(lit(None).cast("string"))
#         .alias("item_discount_name"),
#         expr("""
#             concat(
#                 coalesce(
#                     transform(item.discounts, x -> named_struct(
#                         'offer_name', x.offerName,
#                         'discount_amount', CAST(x.discountAmount AS DECIMAL(18,3))
#                     )),
#                     array()
#                 ),
#                 CASE WHEN item.billDiscountAmount IS NOT NULL AND item.billDiscountAmount != 0
#                      THEN array(named_struct(
#                             'offer_name', CAST(NULL AS STRING),
#                             'discount_amount', CAST(item.billDiscountAmount AS DECIMAL(18,3))
#                           ))
#                      ELSE CAST(array() AS ARRAY<STRUCT<offer_name:STRING, discount_amount:DECIMAL(18,3)>>)
#                 END
#             )
#         """).alias("item_discount_bifurcate"),   # full breakdown: every discount entry + bill discount, each kept separate
#         col("item.subtotal").cast("decimal(18,3)").alias("net_item_amount"),
#         expr("aggregate(item.taxes.tax_amount, 0D, (acc, x) -> acc + x)")
#             .cast("decimal(18,3)").alias("tax_amount"),
#         col("item.subTotalWithTax").cast("decimal(18,3)").alias("gross_item_amount"),
#         "deployment_name", "business_date", "close_time", "created_time",
#         lit(None).cast("string").alias("parent_upcode"), "order_type", "tab_name",
#         lit(True).alias("is_base_item"),
#         col("item.category.isCombos").alias("is_combo_item"),
#         col("item.category.isAddons").alias("is_addon_item"),
#         lit(False).alias("is_combo_child_item"), lit(False).alias("is_addon_child_item"),
#         lit("POSIST").alias("source_system")
#     )
# )

# addon_items_df = (
#     items_df.filter((col("kot.isVoid") == False) & col("item.addOns").isNotNull()
#                      & (size(col("item.addOns")) > 0))
#     .select("sales_bill_id", "kot_number", "deployment_name", "business_date",
#             "close_time", "created_time", "order_type", "tab_name",
#             col("item.number").alias("parent_upcode"),
#             col("item.itemIndex").alias("item_index"),
#             explode(col("item.addOns")).alias("addon"))
#     .select(
#         "sales_bill_id", "kot_number", "item_index",
#         col("addon.displayName").alias("item_name"),
#         col("addon.category.categoryName").alias("category_name"),
#         col("addon.category.supercategory.supercategoryName").alias("supercategory_name"),
#         col("addon.number").alias("item_upccode"),
#         col("addon.quantity").cast("int").alias("quantity"),
#         col("addon.rate").cast("decimal(18,3)").alias("unit_price"),
#         col("addon.subtotal").cast("decimal(18,3)").alias("net_item_amount"),
#         expr("aggregate(addon.taxes.tax_amount, 0D, (acc, x) -> acc + x)")
#             .cast("decimal(18,3)").alias("tax_amount"),
#         col("addon.subTotalWithTax").cast("decimal(18,3)").alias("gross_item_amount"),
#         col("addon.billDiscountAmount")
#             .cast("decimal(18,3)").alias("item_discount_amount"),
#         lit(None).cast("string").alias("item_discount_name"),
#         when(
#             col("addon.billDiscountAmount").isNotNull() & (col("addon.billDiscountAmount") != 0),
#             expr("array(named_struct('offer_name', CAST(NULL AS STRING), 'discount_amount', CAST(addon.billDiscountAmount AS DECIMAL(18,3))))")
#         ).otherwise(
#             expr("CAST(array() AS ARRAY<STRUCT<offer_name:STRING, discount_amount:DECIMAL(18,3)>>)")
#         ).alias("item_discount_bifurcate"),   # addon.discounts is always null/empty in practice, so this mirrors billDiscountAmount only
#         "deployment_name", "business_date", "close_time", "created_time", "parent_upcode",
#         "order_type", "tab_name",
#         lit(False).alias("is_base_item"), lit(False).alias("is_combo_item"),
#         lit(False).alias("is_addon_item"), lit(False).alias("is_combo_child_item"),
#         lit(True).alias("is_addon_child_item"), lit("POSIST").alias("source_system")
#     )
# )

# all_items_df = base_items_df.unionByName(
#     addon_items_df.select(*base_items_df.columns)   # column order/name aligned union
# )

# COMMAND ----------

# # Databricks notebook source
# from pyspark.sql.functions import col, lit, expr, when, explode, size, to_date, to_timestamp

# # Read source bronze table, dedupe by ng_id
# df = (
#     incremental_df
#     .filter(col("isVoid") == False)
#     .dropDuplicates(subset=["ng_id"])
# )

# # Explode KOTs -> items
# items_df = (
#     df.select(
#         col("ng_id").alias("sales_bill_id"), col("deployment_name"),
#         to_date(col("businessDate")).alias("business_date"),
#         col("tabType").alias("order_type"), col("tab").alias("tab_name"),
#         to_timestamp(col("_closeTime")).alias("close_time"),
#         to_timestamp(col("_created")).alias("created_time"),
#         explode(col("_kots")).alias("kot")
#     )
#     .select(
#         "sales_bill_id", col("kot.kotNumber").alias("kot_number"),
#         "deployment_name", "business_date", "order_type", "tab_name",
#         "close_time", "created_time",
#         explode(col("kot.items")).alias("item")
#     )
# )


# base_items_df = (
#     items_df.filter(col("kot.isVoid") == False)
#     .select(
#         "sales_bill_id", "kot_number",
#         col("item.itemIndex").alias("item_index"),
#         col("item.displayName").alias("item_name"),
#         col("item.category.categoryName").alias("category_name"),
#         col("item.category.supercategory.supercategoryName").alias("supercategory_name"),
#         col("item.number").alias("item_upccode"),
#         col("item.quantity").cast("int").alias("quantity"),
#         col("item.rate").cast("decimal(18,3)").alias("unit_price"),
#         when(col("item.discounts").isNotNull() & (expr("size(item.discounts)") > 0),
#              expr("item.discounts[0].discountAmount"))          # first discount only
#         .otherwise(col("item.billDiscountAmount"))
#         .cast("decimal(18,3)").alias("item_discount_amount"),
#         when(col("item.discounts").isNotNull() & (expr("size(item.discounts)") > 0),
#              expr("item.discounts[0].offerName"))                # first discount's offer name
#         .otherwise(lit(None).cast("string"))
#         .alias("item_discount_name"),
#         col("item.subtotal").cast("decimal(18,3)").alias("net_item_amount"),
#         expr("aggregate(item.taxes.tax_amount, 0D, (acc, x) -> acc + x)")
#             .cast("decimal(18,3)").alias("tax_amount"),
#         col("item.subTotalWithTax").cast("decimal(18,3)").alias("gross_item_amount"),
#         "deployment_name", "business_date", "close_time", "created_time",
#         lit(None).cast("string").alias("parent_upcode"), "order_type", "tab_name",
#         lit(True).alias("is_base_item"),
#         col("item.category.isCombos").alias("is_combo_item"),
#         col("item.category.isAddons").alias("is_addon_item"),
#         lit(False).alias("is_combo_child_item"), lit(False).alias("is_addon_child_item"),
#         lit("POSIST").alias("source_system")
#     )
# )

# addon_items_df = (
#     items_df.filter((col("kot.isVoid") == False) & col("item.addOns").isNotNull()
#                      & (size(col("item.addOns")) > 0))
#     .select("sales_bill_id", "kot_number", "deployment_name", "business_date",
#             "close_time", "created_time", "order_type", "tab_name",
#             col("item.number").alias("parent_upcode"),
#             col("item.itemIndex").alias("item_index"),
#             explode(col("item.addOns")).alias("addon"))
#     .select(
#         "sales_bill_id", "kot_number", "item_index",
#         col("addon.displayName").alias("item_name"),
#         col("addon.category.categoryName").alias("category_name"),
#         col("addon.category.supercategory.supercategoryName").alias("supercategory_name"),
#         col("addon.number").alias("item_upccode"),
#         col("addon.quantity").cast("int").alias("quantity"),
#         col("addon.rate").cast("decimal(18,3)").alias("unit_price"),
#         col("addon.subtotal").cast("decimal(18,3)").alias("net_item_amount"),
#         expr("aggregate(addon.taxes.tax_amount, 0D, (acc, x) -> acc + x)")
#             .cast("decimal(18,3)").alias("tax_amount"),
#         col("addon.subTotalWithTax").cast("decimal(18,3)").alias("gross_item_amount"),
#         col("addon.billDiscountAmount")
#             .cast("decimal(18,3)").alias("item_discount_amount"),
#         lit(None).cast("string").alias("item_discount_name"),
#         "deployment_name", "business_date", "close_time", "created_time", "parent_upcode",
#         "order_type", "tab_name",
#         lit(False).alias("is_base_item"), lit(False).alias("is_combo_item"),
#         lit(False).alias("is_addon_item"), lit(False).alias("is_combo_child_item"),
#         lit(True).alias("is_addon_child_item"), lit("POSIST").alias("source_system")
#     )
# )

# all_items_df = base_items_df.unionByName(
#     addon_items_df.select(*base_items_df.columns)   # column order/name aligned union
# )

# COMMAND ----------

# # Item-
# df_items = all_items_df.select(
#     col("sales_bill_id").alias("ng_id"),
#     "kot_number", "item_index", "deployment_name", "business_date",
#     "order_type", "tab_name", "close_time", "created_time",
#     "item_upccode", "quantity", "unit_price", "net_item_amount",
#     col("tax_amount").alias("item_tax_amount"),
#     "gross_item_amount", "item_discount_amount", "item_discount_name",
#     "item_discount_bifurcate",   # NEW
#     "parent_upcode",
#     "is_base_item", "is_combo_item", "is_addon_item",
#     "is_combo_child_item", "is_addon_child_item",
#     "category_name", "item_name", "supercategory_name", "source_system"
# )

# # Fill missing discounts with 0 so downstream arithmetic doesn't null out
# df_items = df_items.fillna(value=0.000, subset=["item_discount_amount"])

# # Recompute net/gross amounts around the (now reliable) item_discount_amount
# df_items = (
#     df_items
#     .withColumn("net_item_amount_after_discount", col("net_item_amount") - col("item_discount_amount"))
#     .withColumn("gross_item_amount", col("net_item_amount_after_discount") + col("item_tax_amount"))
#     .withColumnRenamed("net_item_amount", "net_item_amount_before_discount")
# )

# COMMAND ----------

# from pyspark.sql.functions import from_utc_timestamp, to_timestamp, when

# # Extract online order source fields from bronze
# df_online_bills = (
#     df.filter(col("onlineBill").isNotNull())
#     .select("ng_id", "onlineBill")
#     .withColumn("source_online_name", col("onlineBill.source.name"))
#     .withColumn("source_online_id", col("onlineBill.source.id"))
#     .withColumn("source_online_order_id", col("onlineBill.source.order_id"))
# )

# df_bills = (
#     spark.read.table("fq_dev_catalog.silver.sales_bills")
#     .join(df_online_bills.select("ng_id", "source_online_name", "source_online_order_id"),
#           on="ng_id", how="left")
#     .drop("sys_id", "business_date", "deployment_name", "order_type",
#           "tab_name", "source_system", "close_time")
# )

# df_items_bills = df_items.join(df_bills, on="ng_id", how="left")

# df_items_bills_final = (
#     df_items_bills
#     .withColumn("created_time", from_utc_timestamp(to_timestamp(col("created_time")), "Asia/Dubai"))
#     .withColumn("close_time", from_utc_timestamp(to_timestamp(col("close_time")), "Asia/Dubai"))
#     .withColumn("source_system", when(col("source_online_name").isNotNull(), "ONLINE").otherwise(col("source_system")))
# )

# COMMAND ----------

# df_final = df_items_bills_final.select(
#     "deployment_name", "business_date", "company_id", "country_code", "ng_id", "bill_id",
#     "kot_number",  "source_system", "source_online_name", "source_online_order_id",
#     "item_upccode", "quantity", "unit_price",
#     "net_item_amount_before_discount", "item_discount_amount", "item_discount_name",
#     "item_discount_bifurcate",   # NEW
#     "net_item_amount_after_discount", "item_tax_amount", "gross_item_amount",
#     "parent_upcode", "order_type", "tab_name",
#     "is_base_item", "is_combo_item", "is_addon_item", "is_combo_child_item", "is_addon_child_item",
#     "category_name", "item_name", "supercategory_name", "created_time", "close_time",
#     "kots_ids",
#     col("net_amount_before_discount").alias("bill_net_amount_before_discount"),
#     "bill_discount", "total_items_discount",
#     col("net_amount_after_discount").alias("bill_net_amount_after_discount"),
#     col("tax_amount").alias("bill_tax_amount"),
#     col("gross_amount").alias("bill_gross_amount"),
#     "total_items", "is_void", "void_reason", "covers"
# )

# df_final.display()

# COMMAND ----------

# One-time: add the new columns to the existing gold table (run once, then re-comment)
# spark.sql("""
# ALTER TABLE fq_dev_catalog.gold.itemwise_billwise_fact ADD COLUMNS (
#     bill_payments_type   ARRAY<STRING>,
#     bill_payments_name   ARRAY<STRING>,
#     bill_payments_amount ARRAY<DECIMAL(18,3)>
# )
# """)

# COMMAND ----------

# DBTITLE 1,Cell 10
# from delta.tables import DeltaTable

# TARGET_TABLE = "fq_dev_catalog.gold.itemwise_billwise_fact"

# target = DeltaTable.forName(spark, TARGET_TABLE)

# (
#     target.alias("tgt")
#     .merge(
#         df_final.alias("src"),
#         """
#         tgt.ng_id = src.ng_id
#         AND tgt.kot_number = src.kot_number
#         AND tgt.item_upccode = src.item_upccode
#         AND COALESCE(tgt.parent_upcode, '') = COALESCE(src.parent_upcode, '')
#         """
#     )
#     .whenMatchedDelete()
#     .execute()
# )

# # COMMAND ----------

# df_final.write.format("delta") \
#     .mode("append") \
#     .saveAsTable(TARGET_TABLE)

# print("Inserted fresh rows into silver")

# COMMAND ----------


# from pyspark.sql.functions import current_timestamp

# SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
# CONSUMER_NAME = "new_billwise_itemwise_fact"
# VERSION_TABLE = "fq_dev_catalog.bronze.version_control"

# spark.sql(f"""
# UPDATE {VERSION_TABLE}
# SET
#   last_processed_version = {latest_version},
#   updated_at = current_timestamp()
# WHERE
#   source_table = '{SOURCE_TABLE}'
#   AND consumer_name = '{CONSUMER_NAME}'
# """)

# COMMAND ----------

# from pyspark.sql.functions import col
# spark.read.table("fq_dev_catalog.gold.itemwise_billwise_fact").filter(col("business_date") == "2026-08-17").filter(col("deployment_name") == 'Dennys ADM').count()


# COMMAND ----------

# %sql
# CREATE TABLE fq_dev_catalog.gold.itemwise_billwise_fact (
#     deployment_name                    STRING,
#     business_date                      DATE,
#     company_id                         STRING,
#     country_code                       STRING,
#     ng_id                               STRING,
#     bill_id                             STRING,
#     kot_number                          BIGINT,
#     source_system                       STRING NOT NULL,
#     source_online_name                  STRING,
#     source_online_order_id              STRING,
#     item_upccode                        STRING,
#     quantity                            INT,
#     unit_price                          DECIMAL(18,3),
#     net_item_amount_before_discount     DECIMAL(18,3),
#     item_discount_amount                DECIMAL(18,3),
#     item_discount_name                  STRING,
#     net_item_amount_after_discount      DECIMAL(19,3),
#     item_tax_amount                     DECIMAL(18,3),
#     gross_item_amount                   DECIMAL(20,3),
#     parent_upcode                       STRING,
#     order_type                          STRING,
#     tab_name                            STRING,
#     is_base_item                        BOOLEAN NOT NULL,
#     is_combo_item                       BOOLEAN,
#     is_addon_item                       BOOLEAN,
#     is_combo_child_item                 BOOLEAN NOT NULL,
#     is_addon_child_item                 BOOLEAN NOT NULL,
#     category_name                       STRING,
#     item_name                           STRING,
#     supercategory_name                  STRING,
#     created_time                        TIMESTAMP,
#     close_time                          TIMESTAMP,
#     kots_ids                            ARRAY<BIGINT>,
#     bill_net_amount_before_discount     DECIMAL(18,3),
#     bill_discount                       DECIMAL(18,3),
#     total_items_discount                DECIMAL(18,3),
#     bill_net_amount_after_discount      DECIMAL(18,3),
#     bill_tax_amount                     DECIMAL(18,3),
#     bill_gross_amount                   DECIMAL(18,3),
#     total_items                         INT,
#     is_void                             BOOLEAN,
#     void_reason                         STRING,
#     covers                              INT,
#     bill_payments_type                  ARRAY<STRING>,
#     bill_payments_name                  ARRAY<STRING>,
#     bill_payments_amount                ARRAY<DECIMAL(18,3)>
# )
# USING DELTA
# PARTITIONED BY (business_date)
# LOCATION 'abfss://fq-dev-gold-container@fqadfstoragedev.dfs.core.windows.net/external_tables/billwise_itemwise_fact/'
# COMMENT 'Itemwise-billwise fact table capturing line-item level billing transactions';

# COMMAND ----------



# COMMAND ----------

# spark.read.table("fq_dev_catalog.silver.sales_bills") \
#     .filter(col("ng_id") == "9f9fe1fc-f8ee-7e16-c803-52f7e90b634c-7dae4819-4136-ff1d1a01528c-348672") \
#     .show(truncate=False)

In [0]:
%pip install openpyxl

In [0]:
%restart_python

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS fq_dev_catalog.silver.exports")

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import shutil

SRC = "fq_dev_catalog.gold.sales_orders_payload_1"
LOCAL = "/tmp/sales_orders_payload.xlsx"
OUT = "/Volumes/fq_dev_catalog/silver/exports/sales_orders_payload.xlsx"

df = spark.table(SRC)

# ---- Items: one row per item ----
df_items = df.select(
    F.col("id").alias("order_id"),
    F.col("branch.id").alias("branch_code"),
    F.col("branch.name").alias("branch_name"),
    "salesTypeCode", "placedAt",
    F.explode_outer("items").alias("it")
).select(
    "order_id", "branch_code", "branch_name", "salesTypeCode", "placedAt",
    F.col("it.id").alias("item_id"),
    F.col("it.name").alias("item_name"),
    F.col("it.quantity").alias("quantity"),
    F.round("it.unitPrice.excludingTax", 3).alias("unit_price_ex_tax"),
    F.round("it.unitPrice.discountAmount", 3).alias("unit_discount"),
    F.round("it.unitPrice.includingTax", 3).alias("unit_price_inc_tax"),
    F.round(F.col("it.quantity") * F.col("it.unitPrice.includingTax"), 3).alias("line_total_inc_tax"),
    F.size("it.modifiers").alias("modifier_count"),
)

# ---- Modifiers: one row per modifier ----
df_mods = df.select(
    F.col("id").alias("order_id"),
    F.col("branch.name").alias("branch_name"),
    "salesTypeCode",
    F.explode("items").alias("it")
).select(
    "order_id", "branch_name", "salesTypeCode",
    F.col("it.id").alias("item_id"),
    F.col("it.name").alias("item_name"),
    F.explode("it.modifiers").alias("m")
).select(
    "order_id", "branch_name", "salesTypeCode", "item_id", "item_name",
    F.col("m.id").alias("modifier_id"),
    F.col("m.name").alias("modifier_name"),
    F.col("m.quantity").alias("quantity"),
    F.round("m.unitPrice.excludingTax", 3).alias("unit_price_ex_tax"),
    F.round("m.unitPrice.discountAmount", 3).alias("unit_discount"),
    F.round("m.unitPrice.includingTax", 3).alias("unit_price_inc_tax"),
)

# ---- Orders: one row per payload, with totals ----
df_totals = df_items.groupBy("order_id").agg(
    F.countDistinct("item_id").alias("distinct_items"),
    F.sum("quantity").alias("total_item_qty"),
    F.sum("line_total_inc_tax").alias("total_inc_tax"),
)

df_orders = df.select(
    F.col("id").alias("order_id"),
    F.col("branch.id").alias("branch_code"),
    F.col("branch.name").alias("branch_name"),
    "salesTypeCode", "placedAt", "push_status",
).join(df_totals, "order_id", "left") \
 .orderBy("branch_name", "salesTypeCode")

# ---- Write Excel: local first, then copy to volume ----
with pd.ExcelWriter(LOCAL, engine="openpyxl") as xw:
    df_orders.toPandas().to_excel(xw, sheet_name="Orders", index=False)
    df_items.orderBy("branch_name", "salesTypeCode", "item_name").toPandas().to_excel(xw, sheet_name="Items", index=False)
    df_mods.orderBy("branch_name", "salesTypeCode", "item_name").toPandas().to_excel(xw, sheet_name="Modifiers", index=False)

shutil.copyfile(LOCAL, OUT)
print("Saved:", OUT)

In [0]:
%sql
select distinct deployment_name from fq_dev_catalog.gold.sales_daily_performance

In [0]:
# %sql
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark.sql("select * from fq_dev_catalog.gold.sales_daily_performance where calendar_date = '2026-10-01'").filter(F.col("deployment_name")=="Dennys DCC").display()




In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("ng_id").orderBy(F.col("load_datetime").desc())

df_deduped = spark.table("fq_dev_catalog.bronze.sales_bills_2") \
    .filter(F.col("businessDate") == "2026-09-23") \
    .filter(F.col("deployment_name")=="ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 1005004") \
    .filter(F.col("isVoid") == False) \
    .withColumn("rn", F.row_number().over(w)) \
    .filter(F.col("rn") == 1)

In [0]:
df_items = df_deduped.select(
    "ng_id", "_isDelivery", "_isTakeOut",
    F.explode("_kots").alias("kot")
).select(
    "ng_id","_isDelivery","_isTakeOut",
    F.explode("kot.items").alias("item")
)
df_items.display()

In [0]:
df_base_qty = df_items.groupBy(F.col("item.number").alias("item_code")).agg(F.sum("item.quantity").alias("posist_qty"))

df_base_qty.display()

In [0]:
df_combo_component_qty = df_items.select(
    F.col("item.quantity").alias("combo_item_qty"),
    F.explode("item.mapComboItems").alias("combo")
)
df_combo_component_qty.display()

In [0]:
df_combo_component_qty = df_combo_component_qty.groupBy(
    F.col("combo.number").alias("item_code")
).agg(
    F.sum(F.col("combo.quantity") * F.col("combo_item_qty")).alias("posist_combo_qty")
)

# df_combo_component_qty.display()
df_combo_component_qty.display()

In [0]:
df_addon_qty = df_items.select(
    F.explode("item.addOns").alias("addon")
)
df_addon_qty.display()

In [0]:
df_addon_qty = df_addon_qty.groupBy(
    F.col("addon.number").alias("item_code")
).agg(
    F.sum("addon.quantity").alias("posist_addon_qty")
)
df_addon_qty.display()

##Step 4 — Combine all three sources into one true total per item code (this is your "POSIST Total Qty" reference number)


In [0]:
df_posist_total = df_base_qty.join(df_combo_component_qty,"item_code","full_outer")
df_posist_total.display()

In [0]:
df_a = df_base_qty.alias("a")
df_b = df_combo_component_qty.alias("b")
df_c = df_addon_qty.alias("c")

df_posist_total = df_a.join(df_b, "item_code", "full_outer") \
    .join(df_c, "item_code", "full_outer") \
    .withColumn("posist_total_qty",
        F.coalesce(F.col("posist_qty"), F.lit(0)) +
        F.coalesce(F.col("posist_combo_qty"), F.lit(0)) +
        F.coalesce(F.col("posist_addon_qty"), F.lit(0))
    ).select("item_code", "posist_total_qty")

display(df_posist_total.orderBy("item_code"))

In [0]:
df_supy_qty = spark.table("fq_dev_catalog.gold.sales_orders_payload_1")\
    .filter(F.col("branch.name")=="ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 1005004")\
        .select(F.explode("items").alias("item"))\
            .select(F.col("item.id").alias("item_code"),F.col("item.name").alias("item_name"),F.col("item.quantity").alias("item_qty"))

df_supy_qty.display()



In [0]:
df_supy_mod_qty = spark.table("fq_dev_catalog.gold.sales_orders_payload_1")\
    .filter(F.col("branch.name")=="ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 1005004")\
        .select(F.explode("items").alias("item"))\
            .select(F.explode("item.modifiers").alias("mod"))\
                .select(F.col("mod.id").alias("item_code"),F.col("mod.name").alias("item_name"),F.col("mod.quantity").alias("mod_qty"))

df_supy_mod_qty.display()

In [0]:
df_supy_total = df_supy_qty.groupBy("item_code","item_name").agg(F.sum("item_qty").alias("supy_item_qty")) \
    .join(
        df_supy_mod_qty.groupBy("item_code","item_name").agg(F.sum("mod_qty").alias("supy_mod_qty")),
        ["item_code","item_name"], "full_outer"
    ).withColumn("supy_total_qty",
        F.coalesce(F.col("supy_item_qty"), F.lit(0)) + F.coalesce(F.col("supy_mod_qty"), F.lit(0))
    ).select("item_code","item_name", "supy_total_qty")

df_supy_total.display()

In [0]:
df_final_check = df_posist_total.join(df_supy_total, "item_code", "full_outer") \
    .withColumn("qty_variance", F.coalesce(F.col("supy_total_qty"), F.lit(0)) - F.coalesce(F.col("posist_total_qty"), F.lit(0))) \
    .orderBy("item_code")

display(df_final_check)

In [0]:
df_final_check.select("item_code","item_name","supy_total_qty").display()

In [0]:
# ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 1005004 ---->>>> SandboX-2  

![image_1790686086868.png](./image_1790686086868.png "image_1790686086868.png")


#this is where I started Debugging the sales Issue

In [0]:
%sql
SELECT * FROM fq_dev_catalog.information_schema.tables WHERE table_catalog='fq_dev_catalog' AND table_name='sales_daily_performance'

In [0]:
%sql
DESCRIBE HISTORY fq_dev_catalog.gold.sales_daily_performance LIMIT 10

In [0]:
%sql
SELECT
    deployment_name,
    is_void,
    SUM(COALESCE(bill_discount, 0) + COALESCE(total_items_discount, 0)) AS total_discount,
    SUM(net_amount_before_discount) AS net_sales_before_discount,
    SUM(net_amount_after_discount) AS net_sales_after_discount,
    SUM(gross_amount) AS gross_sales
FROM fq_dev_catalog.silver.sales_bills
WHERE business_date = '2026-09-10'
  AND (
        deployment_name LIKE '%AL AIN%'
        OR deployment_name LIKE '%KARAMA%'
      )
GROUP BY
    deployment_name,
    is_void;

In [0]:
%sql
WITH deduped AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY ng_id ORDER BY load_datetime DESC) as rn
  FROM fq_dev_catalog.bronze.sales_bills_2
  WHERE businessDate = '2026-09-10'
  AND (deployment_name LIKE '%AL AIN%' OR deployment_name LIKE '%KARAMA%')
  AND (isVoid = false OR isVoid IS NULL)
)
SELECT deployment_name,
  SUM(aggregation.totalAmount) as net_before_discount,
  SUM(coalesce(aggregation.billDiscount,0)) as bill_discount,
  SUM(coalesce(aggregation.totalItemDiscount,0)) as item_discount,
  SUM(aggregation.totalAmount - coalesce(aggregation.billDiscount,0) - coalesce(aggregation.totalItemDiscount,0)) as net_after_discount,
  SUM(aggregation.totalTax) as tax_amount,
  SUM(aggregation.netAmount) as gross_amount
FROM deduped
WHERE rn = 1
GROUP BY deployment_name

In [0]:
from pyspark.sql import functions as F

df = spark.table("fq_dev_catalog.gold.sales_orders_payload_1")

#filter branch

df_filtered = df.filter(F.col("branch.name")=="ALBAIK - SQ DB11 - AL AIN - 1005011")

#exploded items 
df_items = df_filtered.select(
    F.explode(F.col("items")).alias("item")
)

df_items.display()

In [0]:
# Calculate item-level totals
item_totals = df_items.agg(
    F.sum(
        F.col("item.unitPrice.includingTax") * F.col("item.quantity")
    ).alias("item_incl_total"),

    F.sum(
        F.col("item.unitPrice.excludingTax") * F.col("item.quantity")
    ).alias("item_excl_total"),

    F.sum(
        F.col("item.unitPrice.discountAmount") * F.col("item.quantity")
    ).alias("item_disc_total")
)

# Explode modifiers
df_modifiers = df_items.select(
    F.explode("item.modifiers").alias("mod")
)

# Calculate modifier total
mod_totals = df_modifiers.agg(
    F.sum(
        F.col("mod.unitPrice.includingTax") * F.col("mod.quantity")
    ).alias("mod_incl_total")
)

In [0]:
# Combine item and modifier totals
result = item_totals.crossJoin(mod_totals)

result.display(truncate=False)

In [0]:
+# Al Ain - 25329.48

In [0]:
%sql
SELECT salesTypeCode, COUNT(*) AS rows_cnt
FROM fq_dev_catalog.gold.sales_orders_payload_1
GROUP BY salesTypeCode;

In [0]:
%sql
UPDATE fq_dev_catalog.gold.sales_orders_payload_1
SET salesTypeCode = CASE lower(trim(salesTypeCode))
    WHEN 'dine in'  THEN 'DI'
    WHEN 'delivery' THEN 'DLY'
    WHEN 'takeaway' THEN 'TA'
    ELSE salesTypeCode
END
WHERE lower(trim(salesTypeCode)) IN ('dine in', 'delivery', 'takeaway');

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_orders_payload_1
    